# Clasificador usando el algortimo de Árbol de Decisión (_Decision Tree_)


## 1. Importar las librerías


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_iris
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 2. Cargar el Dataset Iris


El conjunto de datos IRIS es probablemente el primer conjunto de datos que los estudiantes y principiantes experimentan mientras aprenden ciencia de datos y aprendizaje automático (_machine learning_). El conjunto de datos IRIS es simple y fácil de entender, y ha sido utilizado por estudiantes, investigadores y profesionales durante décadas para aprender los conceptos básicos de clasificación y análisis de datos.

El conjunto de datos IRIS fue publicado en 1936 por el estadístico británico Ronald A. Fisher en el artículo “**El uso de mediciones múltiples en problemas taxonómicos**” y, desde entonces, el conjunto de datos IRIS se ha hecho conocido como el “Hola mundo” del aprendizaje automático (ML) porque proporciona una introducción muy efectiva a los conjuntos de datos y algoritmos del mundo real para los estudiantes.

![Descripción](iris.png)

El conjunto de datos Iris es un conjunto de datos de **150 filas** y **4 características** de medidas de flores de iris (**longitud del sépalo, ancho del sépalo, longitud de los pétalos, ancho de los pétalos**) en **tres especies** — **Setosa, Versicolor y Virginica**. La longitud y el ancho de los pétalos son las dos características más correlacionadas con las especies, lo que las convierte en los predictores más fuertes. Es el primer conjunto de datos estándar para aprender la clasificación porque es pequeño, limpio y no requiere limpieza de datos antes del modelado.

Las cuatro características medidas (en centímetros) para cada flor:
Estas son las medidas que utilizamos para hacer nuestra predicción. Puedes pensar en ellos como las piezas de un rompecabezas.

- Longitud del sépalo (cm): El sépalo es la parte de la flor que protege el brote antes de florecer. Esta característica es una medida de la longitud del sépalo.
- Ancho del sépalo (cm): es una medida del ancho del sépalo.
- Longitud del pétalo (cm): El pétalo es la parte muy colorida de la flor. Esta es una medida de cuánto tiempo dura.
- Ancho del pétalo (cm): es una medida del ancho del pétalo.

![Descripción](iris_dimensions.png)

Tomado de: [Guvi](https://www.guvi.in/blog/iris-dataset-explained-features/)

Las clases objetivo (los resultados):

- Iris Setosa (0): Esta especie se distingue claramente de las otras dos especies de Iris, por lo que puede considerarse como el "modo fácil" de este conjunto de datos para un clasificador.
- Iris Versicolor (1): Esta especie es muy similar a Virginica, lo que significa que es más difícil distinguirlas. ¡Ahí radica la verdadera dificultad!
- Iris Virginica (2): La tercera especie que es más probable que se confunda con Versicolor basándose únicamente en las medidas de dimensiones.


In [ ]:
# Cargar el dataset
# iris = load_iris()
# iris_df = pd.DataFrame(iris.data, columns=iris.feature_names)

# iris_df = iris.frame
# iris_df.to_csv('./dataset/iris.csv')
# iris_df = pd.read_csv('..\\datasets\\iris.csv', header=0, index_col=0) # Ruta relativa para Windows
iris_df = pd.read_csv(
    "./dataset/iris.csv", header=0, index_col=0
)  # Ruta relativa MacOS y Linux

In [ ]:
# Nombres de las características
# iris.feature_names

In [ ]:
iris_df.columns

## 3. Análisis Exploratorio de Datos (EDA)


In [ ]:
# Dimensiones del dataset
iris_df.shape

In [ ]:
# Tipos de datos del dataset
iris_df.dtypes

In [ ]:
# Primeros registros del dataset
iris_df.head(5)

In [ ]:
# Últimos registros del dataset
iris_df.tail(5)

In [ ]:
# Descripción del dataset
iris_df.describe()

In [ ]:
# Iris Setosa(0)
iris_df[iris_df.target == 0].describe()

In [ ]:
# Iris Versicolor(1)
iris_df[iris_df.target == 1].describe()

In [ ]:
# Iris Virginica(2)
iris_df[iris_df.target == 2].describe()

In [ ]:
iris_df.hist()

In [ ]:
# Histograma para la clase setosa (0)
iris_df[iris_df.target == 0].hist()

In [ ]:
# Histograma para la clase Versicolor (1)
iris_df[iris_df.target == 1].hist()

In [ ]:
# Histograma para la clase Virginica (2)
iris_df[iris_df.target == 2].hist()

In [ ]:
iris_df["target"].value_counts().plot(kind="bar")
plt.title("Distribución de las especies")
plt.xlabel("Especie (Setosa, Versicolor, Virginica)")
plt.ylabel("Frecuencia")
plt.show()

## Verificar valores faltantes


In [ ]:
iris_df.isnull().sum()

In [ ]:
print("Porcentaje de valores nulos ")
iris_df.isnull().mean() * 100

# Correlación entre variables

- ¿Qué variables mueven el negocio y dónde enfocar los recursos?


In [ ]:
plt.figure(figsize=(10, 8))  # 10 x 8 pulgadas
sns.heatmap(iris_df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Matriz de correlación de características")
plt.tight_layout()
plt.show()

**Insights clave**:

- Fuertemente positivo: (r > 0.8) petal_length <-> petal_width: r = 0.96 multicolineal
- Negativo: (r < -0.3) sepal_width <-> petal_length: r = -0.43

_¿Por qué es importante?_ La correlación solo refleja relaciones lineales. Dos variables pueden estar fuertemente relacionadas de forma no lineal y aun así mostrar una correlación cercana a cero. Además, la correlación no implica causalidad. Las características altamente correlacionadas no aportan información nueva. Considere eliminar una.

- Priorización de Features: Centrar los algoritmos y dashboards de control en las dimensiones de los pétalos (petal length y petal width).
- Reglas de Negocio: Construir árboles de decisión con profundidad reducida (shallow trees), ya que los cortes basados en el pétalo separan limpiamente los segmentos de clientes/productos


## Visualizar las distribuciones


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 10))
axes = axes.flatten()
for i, col in enumerate(iris_df.columns):
    iris_df[col].hist(bins=50, ax=axes[i])
    axes[i].set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
# Mapea los números de target a los nombres reales de las especies
target_names = {0: "Setosa", 1: "Versicolor", 2: "Virginica"}
iris_plot = iris_df.copy()
iris_plot["species"] = iris_plot["target"].map(target_names)

features = [
    "sepal length (cm)",
    "sepal width (cm)",
    "petal length (cm)",
    "petal width (cm)",
]

# Crear cuadrícula de 2x2 gráficos
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
axes = axes.flatten()

for i, col in enumerate(features):
    sns.histplot(
        data=iris_plot,
        x=col,
        hue="species",
        kde=True,  # Agrega curva de densidad
        ax=axes[i],
        palette="tab10",
        element="step",
        alpha=0.6,
    )
    axes[i].set_title(f"Histograma: {col}", fontsize=12, fontweight="bold")
    axes[i].set_xlabel("Medida (cm)")
    axes[i].set_ylabel("Frecuencia")

plt.suptitle(
    "Distribución por Variable según Especie (Iris Dataset)",
    fontsize=15,
    fontweight="bold",
    y=1.02,
)
plt.tight_layout()
plt.show()

## Gráfico de Dispersión con Codificación de colores


In [ ]:
sns.set_theme(style="whitegrid")
sns.scatterplot(
    data=iris_df,
    x="sepal length (cm)",
    y="petal length (cm)",
    hue="target",
    palette="deep",
)
plt.title("Iris Dataset — Sepal vs Petal Length")
plt.show()

## Análisis de Covarianza

- Cuando una empresa busca clasificar clientes o productos en distintos segmentos, la métrica con mayor varianza suele ser la que traza las fronteras más claras. Esta es la dimensión donde se encuentran las verdaderas diferencias entre grupos.
- Permite aplicar técnicas como Análisis de Componentes Principales (PCA) para consolidar 4 indicadores en 1 o 2 índices ejecutivos (KPIs compuestos) sin perder precisión. Optimización de costos.


In [ ]:
# Enfoque Gerencial: Magnitud y dirección conjunta en unidades originales (cm²)
sns.set_theme(style="whitegrid")
# features = ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']

cov_matrix = iris_df[features].cov()
print("Matriz de Covarianza")
print(cov_matrix)

plt.figure(figsize=(8, 6))
sns.heatmap(cov_matrix, annot=True, cmap="Blues", fmt=".3f")
plt.title("Matriz de Covarianza (Variabilidad Conjunta en cm²)", fontweight="bold")
plt.tight_layout()
plt.show()

## Relación de Variables Independientes (predictoras) y Dependiente (predicha)

- Automatización de Decisiones (Regla 80/20): Para clasificar la flor Setosa, no se requiere un servidor de machine learning, una validación condicional (if petal_length <= 2.45 then "Setosa") reduce la carga computacional en un $33%$.
- Concentración del esfuerzo analítico: Los esfuerzos de calibración del modelo deben centrarse en diferenciar entre Versicolor y Virginica (pétalos entre $4.5$ y $5.1\text{ cm}$ de longitud), donde existe la mayor probabilidad de falsos positivos/negativos.
- Racionalización de Recursos: El sepal width de Versicolor ($2.77$) y Virginica ($2.97$) es prácticamente idéntico con desviaciones solapadas. El monitoreo de esta variable no aporta retorno al negocio para diferenciar estas dos categorías.


In [ ]:
# Mapeo de nombres si no existe la columna 'species'
if "species" not in iris_df.columns:
    target_names = {0: "Setosa", 1: "Versicolor", 2: "Virginica"}
    iris_df["species"] = iris_df["target"].map(target_names)

# Gráfico Boxplots para comparar la capacidad discriminante de cada característica
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for i, col in enumerate(features):
    sns.boxplot(
        data=iris_df,
        x="species",
        y=col,
        hue="species",
        ax=axes[i],
        palette="Set2",
        legend=False,
    )
    axes[i].set_title(f"Poder Discriminante: {col}", fontweight="bold")
    axes[i].set_xlabel("Tipo de Flor (Clase)")
    axes[i].set_ylabel("cm")

plt.suptitle(
    "Análisis de variables predictoras y objetivo",
    fontsize=14,
    fontweight="bold",
    y=1.02,
)
plt.tight_layout()
plt.show()

In [ ]:
# Resumen estadístico de la media y la desviación estándar agrupada
business_summary = iris_df.groupby("species")[features].agg(["mean", "std"])
print("Perfil promedio por especie de flor (KPI)")
print(business_summary.round(2))

## 4. Análisis Confirmatorio de Datos (CDA)

#### Se usan para pasar de una observación exploratoria a una conclusión estadística.

### Hipótesis:

#### H0 (Nula): Las dimensiones del pétalo son iguales en promedio entre las 3 especies de flores (no se pueden clasificar).

#### H1 (Alternativa): Existe una diferencia significativa en las dimensiones del pétalo entre especies de flores.


In [ ]:
from scipy import stats

print(f"Contraste de Hipótesis (Anova de un factor)")
alpha = 0.05

for col in ["petal length (cm)", "petal width (cm)"]:
    grupo_setosa = iris_df[iris_df["species"] == "Setosa"][col]
    grupo_versicolor = iris_df[iris_df["species"] == "Versicolor"][col]
    grupo_virginica = iris_df[iris_df["species"] == "Virginica"][col]

    f_stat, p_val = stats.f_oneway(grupo_setosa, grupo_versicolor, grupo_virginica)

    decision = (
        "Rechazar la Hipótesis Nula H0 porque p-value < 0.05"
        if p_val < alpha
        else "No se rechaza H0"
    )
    print(f" Variable: {col}")
    print(f" F-Statistic: {f_stat:.4f} y p-value: {p_val:.4e}")
    print(f" Decisión Operativa: {decision}\n")

#### Prueba de Tukey (HSD - Honestly Significant Difference) dentro del CDA

- El ANOVA no le indica si debe rediseñar la línea de empaque para Setosa, Versicolor o Virginica.
- La prueba de Tukey es la auditoría detallada al comparar los productos o especies (Setosa vs. Versicolor, Setosa vs. Virginica, Versicolor vs. Virginica).
- El valor le indica exactamente dónde sí hay una ventaja o diferencia real medible y dónde hay ambigüedad operativa.
- El control del riesgo corporativo: Si hicieran múltiples comparaciones simples al azar, el "riesgo de falso positivo" se dispararía. Tukey corrige ese sesgo matemático y garantiza que el nivel de confianza del 95% se mantenga.


In [ ]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# Prueba de Tukey (HSD - Honestly Significant Difference) dentro del CDA
variable_analizada = "petal length (cm)"
tukey = pairwise_tukeyhsd(
    endog=iris_df[variable_analizada], groups=iris_df["species"], alpha=0.05
)
print("Reporte ejecutivo de diferencias de especies de flores (Test Tukey HSD)")
print(tukey)

# Gráfica de visualización de intervalos de confianza para las especies de flores
fig = tukey.plot_simultaneous(figsize=(6, 4))
plt.title(
    f"Prueba de Tukey al 95% de Confianza para: {variable_analizada}", fontweight="bold"
)
plt.xlabel("Medida promedio (cm)")
plt.ylabel("Especie de flores")
plt.axvline(x=0, color="red", linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Conclusiones del Análisis Exploratorio y Confirmatorio de Datos

1. Las matrices de covarianza y correlación: Explica que la covarianza mide la dirección en unidades reales ($cm^2$), mientras que la correlación estandariza este valor entre $-1$ y $1$. En decisiones de negocios las correlaciones $\ge 0.90$ (como el largo y ancho de pétalo) implican redundancia; en procesos industriales significa que medir ambos parámetros duplica el costo de captura de datos sin agregar valor discriminante.
2. Los gráficos de Boxplots como herramientas de decisión: Muestran que la longitud y el ancho del pétalo separan la flor de Setosa de las demás especies, mientras que el sépalo presenta solapamiento que puede generar errores de clasificación.
3. El valor del CDA (ANOVA): Con un $p\text{-value} < 0.05$ valida formalmente que el proyecto de Machine Learning se fundamenta en datos estadísticos reales y no en variaciones aleatorias del proceso.
4. Los dos hallazgos de las especies Setosa versus Versicolor y Virginica: La prueba arroja un reject = True contundente con una brecha enorme en pétalo.
5. Decisión para la organización: Automatización de clasificación inmediata a bajo costo, riesgo operativo cero.
6. Versicolor vs. Virginica: Aunque estadísticamente hay diferencia, el margen es mucho más estrecho. Alerta de control de calidad. Aquí es donde ocurrirán las devoluciones o reclamos de clientes si el sensor no tiene suficiente precisión.
